In [ ]:
from com.example.agentic.agent.LLMManager import LLMManager
# 
llm = LLMManager.create_llm('ollama')
llm.call('Hello')

#### LOAD Documents

In [ ]:
from com.example.agentic.loader.LoadManager import LoadManager
# Website Data Ingestion 
documents = LoadManager.from_url(["https://docs.crewai.com/how-to/Installing-CrewAI/"])

#### Chunkings

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

#
print(len(chunks))


##### Embaddings

In [ ]:
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager

#chunks=[doc.page_content for doc in documents]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(documents)
print("[INFO] Example embedding:", embeddings[0] if len(embeddings) > 0 else None)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(documents,embeddings)

In [ ]:
import os
from crewai_tools import SerperDevTool,ScrapeWebsiteTool,TavilySearchTool

# Perform search for provide topic
websearch_tool = SerperDevTool()

#
scrape_tool = ScrapeWebsiteTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key= os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)

#### Crew Knowledge Base

In [ ]:
#
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource
from crewai.knowledge.knowledge_config import KnowledgeConfig
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from com.example.agentic.tools.config import _rag_tool_config

# put in agent config knowledge_config=knowledge_config
knowledge_config = KnowledgeConfig(results_limit=10, score_threshold=0.5)

# Create custom storage with specific embedder
custom_storage = KnowledgeStorage(
    embedder=_rag_tool_config['embedder'],
    collection_name="pdf-documents"
)

text_kw_source = TextFileKnowledgeSource(
    file_paths=[
        "designs.txt", 
        "machine_learning.txt",
        "python_intro.txt"
    ]
)
# json_source.storage = custom_storage
#
from crewai.knowledge.source.json_knowledge_source import JSONKnowledgeSource
json_source = JSONKnowledgeSource(
    file_paths=[
        "designs-research.json"
    ]
)
#json_source.storage = custom_storage

In [13]:
from crewai.knowledge.source.base_knowledge_source import BaseKnowledgeSource
import os,requests,json
from typing import Dict, Any
from pydantic import BaseModel, Field

class ImagesKnowledgeSource(BaseKnowledgeSource):
    """Knowledge source that fetches relavent images url for design topic."""

    doc_path: str = Field(description="Json doc path")
    limit: int = Field(default=10, description="Number of images to fetch")

    def load_content(self) -> Dict[Any, str]:
        """Fetch and format desing images."""
        try:
            #response = requests.get(
            #    f"{self.api_endpoint}?limit={self.limit}"
            #)
            #response.raise_for_status()
            #data = response.json()

            work_dir = os.getenv("WORK_DIR")
            with open(f"{work_dir}/{self.doc_path}", 'r') as file:
                _json = json.load(file)
                
            _images = _json.get('design_reference_images', [])

            formatted_data = self.validate_content(_images)
            return {self.doc_path: formatted_data}
        except Exception as e:
            raise ValueError(f"Failed to fetch space news: {str(e)}")

    def validate_content(self, images: list) -> str:
        """Format articles into readable text."""
        formatted = "Design Reference Images:\n\n"
        for _image in images:
            formatted += f"""
                Title: {_image['title']}
                Url: {_image['url']}
                -------------------"""
        return formatted

    def add(self) -> None:
        """Process and store the articles."""
        content = self.load_content()
        for _, text in content.items():
            chunks = self._chunk_text(text)
            self.chunks.extend(chunks)
        self._save_documents()

    def aadd(self) -> None:
        pass


In [14]:
from crewai.knowledge.storage.knowledge_storage import KnowledgeStorage
from com.example.agentic.tools.config import _rag_tool_config

# Create custom storage with specific embedder
design_image_kstorage = KnowledgeStorage(
    embedder=_rag_tool_config['embedder'],
    collection_name="design_images"
)

# Create knowledge source
desing_images = ImagesKnowledgeSource(
    doc_path="docs/json/designs-research.json",
    storage=design_image_kstorage,
    limit=10
)

#desing_images.add()

In [1]:
import os
import json
from crewai.tools import tool

@tool("Image url extract tool")
def image_fetch_tool(doc_path: str) -> str:
    """Tool to extract image url from json file."""
    tool_output = "Design Reference Images:\n\n"
    try:
        work_dir = os.getenv("WORK_DIR")
        with open(f"{work_dir}/{doc_path}", 'r') as file:
            _json = json.load(file)
            
        _images = _json.get('design_reference_images', [])
        formatted = "Design Reference Images:\n\n"
        for _image in _images:
            tool_output += f"""
                title: {_image['title']}
                url: {_image['url']}
                -------------------"""
    except Exception as e:
        raise ValueError(f"Failed to fetch space news: {str(e)}")
    #
    return tool_output

#result = image_fetch_tool('docs/json/designs-research.json')
#print(result)

#### Agent Response Formatters

In [2]:
from pydantic import BaseModel, Field
from typing import List, Dict

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Webpage information with title and url",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    title: str = Field(description="The title of the image")
    url: str = Field(description="The url of the image")

class ResearchImageOutput(BaseModel):
    topic: str = Field(description="The details about primary topic")    
    images: List[ResearchImage] = Field(description="List of top images on topic")
    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )
    references : List[ResearchImage] = Field(description="References of top images on topic")

##### Design Crew

In [3]:
from datetime import datetime
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import ScrapeWebsiteTool, SerperDevTool
from com.example.agentic.tools.config import _rag_tool_config
from com.example.agentic.agent.LLMManager import LLMManager

@CrewBase
class DesignDevelopment():
    """Latest AI Design Development Crew"""

    agents_config = "config/agents.yaml"
    tasks_config = "config/tasks.yaml"

    @agent
    def content_researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['content_researcher'],
            verbose=True,
            tools=[
                SerperDevTool(),
                ScrapeWebsiteTool()
                # DesignSearchTool()
            ],
            allow_delegation=False,
            #knowledge_sources=[text_kw_source],
            #embedder=_rag_tool_config['embedder'],
            #llm=LLMManager.create_llm()
        )
    
    @agent
    def images_extractor(self) -> Agent:
        return Agent(
            config=self.agents_config['images_extractor'],
            #knowledge_sources=[desing_images],
            verbose=True,
            tools=[image_fetch_tool],
            #embedder=_rag_tool_config['embedder'],
            #llm=LLMManager.create_llm()
        )
    
    @agent
    def reporting_analyst(self) -> Agent:
        return Agent(
            config=self.agents_config['reporting_analyst'],
            verbose=True,
            #llm=LLMManager.create_llm()
        )
    
    @agent
    def formatter(self) -> Agent:
        return Agent(
            config=self.agents_config['formatter'],
            verbose=True,
            #llm=LLMManager.create_llm()
        )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'], 
            output_json=ResearchOutput
        )


    @task
    def extract_images_task(self) -> Task:
        return Task(
            config=self.tasks_config['extract_images_task'],
            output_json=ResearchImageOutput,
            #context=[self.research_task()]
        )
    
    
    @task
    def reporting_task(self) -> Task:
        return Task(
            config=self.tasks_config['reporting_task'],
            output_pydantic=ExecutiveReport
        )

    @task
    def formatting_task(self) -> Task:
        return Task(
            config=self.tasks_config['formatting_task']
        )

	
    @crew
    def crew(self) -> Crew:
        """Creates the DesignDevelopment Crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
            embedder=_rag_tool_config['embedder']
        )

In [4]:
from datetime import datetime

_exe_date = datetime.now().strftime('%Y-%m-%d')
_exe_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f" DesignDevelopment Crew triggered on {_exe_date} with execution id {_exe_id}")

def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': _exe_date,
        'run_id': _exe_id
    }
    DesignDevelopment().crew().kickoff(inputs=inputs)

 DesignDevelopment Crew triggered on 2026-04-26 with execution id 20260426_154513


In [ ]:
run()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: DesignDevelopment                                                                                        │
│  ID: 564b3345-6f17-4ef3-ba7c-22dde4e5d31a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: 44deabe4-f7ad-4c55-83df-98ae01114fb1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│  Task: Conduct a thorough research about Microservice Design Make sure you find minimume 10 relevant and        │
│  accurate content on solution design for Microservice Design.                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_points=[ResearchPoint(topic='Microservice Design', findings='A microservices-based architecture is a  │
│  good fit for this problem', relevance='It allows us to scale our application quickly and efficiently',         │
│  sources=[{'title': 'What is Microservices Architecture?', 'url':                                               │
│  'https://www.infoworld.com/article/3236806/microservices-architecture-how-it-can-help-your-enterprise.html'},  │
│  {'title': 'Microservices: A Guide for Developers', 'url':                                                      │
│  'https://developer.ibm.com/blog/service-designed-microservices-for-devs/'}, {'title': 'Microservice            │
│  Architecture | MDN', 'url':                                                                                    │
│  'https://developer.mozilla.org/en-US/docs/Learn/First_steps/Introduction_to_the_world_of_webdevelopment#Micro  │
│  services'}, {'title': 'Microservices - Open Group', 'url': 'http://pubs.opengroup.org/pub/s%7B6357%7D/'},      │
│  {'title': 'What is Microservices?', 'url': 'https://en.wikipedia.org/wiki/Microservices'}]),                   │
│  ResearchPoint(topic='Monolithic vs Microservice Architecture', findings='Monolithic architecture can be        │
│  beneficial when there are complex, transactional applications that require a single codebase and tight         │
│  coupling between different components', relevance='On the other hand, microservice architecture is better      │
│  suited for small-scale applications or legacy systems where modularity may not be possible',                   │
│  sources=[{'title': 'Monolithic vs Microservices Architectures | InfoQ', 'url':                                 │
│  'https://www.infoq.com/articles/Monolithic-vs-Microservices-Architectures'}, {'title': 'Why Use                │
│  Microservices?', 'url': 'https://www.dZone.com/excerpt/why-use-microservices.html'}]),                         │
│  ResearchPoint(topic='Microservice Communication', findings='The choice of inter-service communication          │
│  protocol will depend on the specific use case and project requirements, but some popular options include       │
│  gRPC, REST, and Message Queueing (e.g., RabbitMQ or Apache Kafka)', relevance='It is essential to consider     │
│  the trade-offs between the quality factors of reliability, performance, and scalability when choosing an       │
│  inter-service communication protocol', sources=[{'title': 'Microservice Communication', 'url':                 │
│  'https://www.infoq.com/articles/microservices-communication-patterns'}]), ResearchPoint(topic='Distributed     │
│  Transactions with Microservices', findings='Implementing distributed transactions with microservices is        │
│  possible, but challenging and may require the use of distributed transaction protocols (e.g., 2PC) or          │
│  eventual consistency models', relevance='However, such implementations should be approached with caution due   │
│  to potential performance and scalability implications', sources=[{'title': 'Microservices for Dummies',        │
│  'url': 'https://martinfowler.com/bliki/MicroservicesForDummies.pdf'}, {'title': 'Building Scalable             │
│  Distributed Systems with Microservices', 'url':                                                                │
│  'http://www.mindorks.in/building-scalable-distributed-

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: research_task                                                                                            │
│  Agent: Content Researcher                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│  Task: Format the executive report into a beautiful markdown document without '```'. Follow these guidelines:   │
│  1. Use proper markdown headers (#, ##, ###) 2. Include emojis from the section_emoji field 3. Format key       │
│  findings and insights as bullet points 4. Add proper spacing and sections breaks 5. Make recommendations       │
│  stand out using blockquotes 6. Ensure the date is properly formatted 7. Add a table of contents at the         │
│  beginning 8. Use horizontal rules (---) to separate major sections 9. Add inline citations for each major      │
│  claim using (Author, Year) format 10. Include a Sources section at the end with numbered references 11. Each   │
│  source should include title, publisher/author, and URL 12. Link inline citations to the corresponding entry    │
│  in the Sources section 13. Ensure atleast one or more reference images include                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Table of Contents**                                                                                          │
│  =================                                                                                              │
│                                                                                                                 │
│  1. [Introduction to Microservice Design](#introduction-to-microservice-design)                                 │
│  2. [Monolithic vs Microservice Architecture](#monolithic-vs-microservice-architecture)                         │
│  3. [Microservice Communication Protocols](#microservice-communication-protocols)                               │
│  4. [Distributed Transactions with Microservices](#distributed-transactions-with-microservices)                 │
│  5. [Conclusion and Recommendations](#conclusion-and-recommendations)                                           │
│                                                                                                                 │
│  **Introduction to Microservice Design**                                                                        │
│  =====================================                                                                          │
│                                                                                                                 │
│  📻 A microservices-based architecture is a good fit for this problem due to its ability to scale quickly and   │
│  efficiently.                                                                                                   │
│                                                                                                                 │
│  ### Key Findings:                                                                                              │
│                                                                                                                 │
│  *   A microservices-based architecture is more suitable for projects requiring scalability.                    │
│  *   Monolithic architecture can be beneficial in complex, transactional applications.                          │
│  *   The choice of inter-service communication protocol depends on the project requirements.                    │
│                                                                                                                 │
│  ### Recommendations:                                                                                           │
│  >                                                                                                              │
│                                                                                                                 │
│  **Blockquote:**                                                                                                │
│  > "The benefits of a microservices-based architecture are substantial. By using this approach, you can create  │
│  large systems that are much simpler to implement and manage."                                                  │
│  >                                                                                                              │
│  >                                                                                                              │
│  \*\*Infoworld\*\*, \*                                  

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: formatting_task                                                                                          │
│  Agent: Technical Writing and Markdown Specialist                                                               │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ f9be6e3c-6385-42af-a6e4-a9e965e8bffd                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/f9be6e3c-6385-42a │
│ f-a6e4-a9e965e8bffd?access_code=TRACE-2c709faab5                             │
│ 🔑 Access Code: TRACE-2c709faab5                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: DesignDevelopment                                                                                        │
│  ID: 564b3345-6f17-4ef3-ba7c-22dde4e5d31a                                                                       │
│  Final Output: **Table of Contents**                                                                            │
│  =================                                                                                              │
│                                                                                                                 │
│  1. [Introduction to Microservice Design](#introduction-to-microservice-design)                                 │
│  2. [Monolithic vs Microservice Architecture](#monolithic-vs-microservice-architecture)                         │
│  3. [Microservice Communication Protocols](#microservice-communication-protocols)                               │
│  4. [Distributed Transactions with Microservices](#distributed-transactions-with-microservices)                 │
│  5. [Conclusion and Recommendations](#conclusion-and-recommendations)                                           │
│                                                                                                                 │
│  **Introduction to Microservice Design**                                                                        │
│  =====================================                                                                          │
│                                                                                                                 │
│  📻 A microservices-based architecture is a good fit for this problem due to its ability to scale quickly and   │
│  efficiently.                                                                                                   │
│                                                                                                                 │
│  ### Key Findings:                                                                                              │
│                                                                                                                 │
│  *   A microservices-based architecture is more suitable for projects requiring scalability.                    │
│  *   Monolithic architecture can be beneficial in complex, transactional applications.                          │
│  *   The choice of inter-service communication protocol depends on the project requirements.                    │
│                                                                                                                 │
│  ### Recommendations:                                                                                           │
│  >                                                                                                              │
│                                                                                                                 │
│  **Blockquote:**                                                                                                │
│  > "The benefits of a microservices-based architecture are substantial. By using this approach, you can create  │
│  large systems that are much simpler to implement and manage."                                                  │
│  >                                                                                                              │
│  >                                                                                                              │
│  \*\*Infoworld\*\*, \*                                 

In [ ]:
from crewai.tools import tool

@tool("Basic Functionality")
def my_simple_tool(question: str) -> str:
    """Tool to enhance the question provided."""
    tool_output = f"enhanced_question {question} with clarifications"
    return tool_output

#### Memory Implementation

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM, Memory
from pydantic import BaseModel, Field
from crewai.rag.embeddings.factory import build_embedder
from com.example.agentic.tools.config import create_memory

memory = create_memory()

# Define a Pydantic model to validate order data
class OrderInputModel(BaseModel):
    order_id: int
    product_name: str
    quantity: int = Field(gt=0, description="Quantity must be greater than zero")

# Define a Pydantic model for the output structure
class OrderAnalysisOutputModel(BaseModel):
    total_orders: int
    top_product: str
    total_quantity: int

# Create an agent to analyze orders
order_analyst = Agent(
    role='Order Analyst',
    goal='Analyze the list of orders and provide a summary report.',
    backstory="You are an expert in processing and analyzing order data.",
    verbose=True,
    memory=memory.scope("/agent/order_analyst"),
    #llm=LLM(model="ollama/llama3.2:latest", base_url="http://localhost:11434")
)

# Define the task using the Pydantic models for input and output validation
order_analysis_task = Task(
    description="Analyze the incoming list of orders and provide a report on total orders, top product, and total quantity.",
    expected_output="provide a report on total orders, top product, and total quantity.",
    input_model=OrderInputModel,  # Pydantic model for input validation
    output_model=OrderAnalysisOutputModel,  # Pydantic model for output structure
    agent=order_analyst
)

# Define the crew and process
order_analysis_crew = Crew(
    agents=[order_analyst],
    tasks=[order_analysis_task],
    process=Process.sequential,
    memory=memory,
    #llm=LLM(model="ollama/llama3.2:latest", base_url="http://localhost:11434")
)

# Sample input data (list of orders)
input_data = [
    {"order_id": 1, "product_name": "Laptop", "quantity": 2},
    {"order_id": 2, "product_name": "Monitor", "quantity": 5},
    {"order_id": 3, "product_name": "Laptop", "quantity": 3}
]

# Kick off the task
result = order_analysis_crew.kickoff(inputs={'input_data': input_data})

# The result will be validated and structured by the output_model
print(result)

In [ ]:
from typing import List
from crewai import Agent, Task, Crew, Process, LLM, Memory
from pydantic import BaseModel, Field
from crewai.rag.embeddings.factory import build_embedder
from com.example.agentic.tools.config import create_memory
from crewai.tools import tool

#
memory = create_memory()

@tool("Sum Function")
def my_simple_tool(question: str) -> str:
    """Tool to perform sum for json attribute."""
    _json = json.loads(question)
    print(_json)
    tool_output = f"enhanced_question {question} with clarifications"
    return tool_output

# 1. Define Pydantic Model for JSON data
class Item(BaseModel):
    name: str
    price: float

class ItemsList(BaseModel):
    items: List[Item]

# 2. Define the Agent
analyst_agent = Agent(
    role='Data Analyst',
    goal='Calculate the total price of items by name',
    backstory='You are expert at analyzing JSON data and calculating sums.',
    memory=memory, # Enable Agent Memory
    verbose=True
)

# 3. Define the Task with Output Model
totaling_task = Task(
    description='Analyze the items and calculate the total price by name.',
    expected_output='The total sum of price by name.',
    agent=analyst_agent,
    tools=[my_simple_tool],
    output_pydantic=ItemsList # Ensures structured JSON
)

# 4. Define Crew with Memory
crew = Crew(
    agents=[analyst_agent],
    tasks=[totaling_task],
    memory=memory,
    process=Process.sequential
)

input_data = [
    {"name":"Laptop",  "price": 2.50},
    {"name":"Monitor", "price": 5.25},
    {"name":"Laptop",  "price": 3.75}
]

# 5. Run and retrieve result
result = crew.kickoff(inputs={'input_data': input_data})
print(result)


####
####